# 3 · Repeater chain on FABRIC — swapping across three real nodes

The FABRIC counterpart to `12_repeater`. The repeater **station runs on the switch
node**, physically between the endpoints, so all three classical links traverse real
WAN segments — in particular the heralds cross the station→bob segment on their own
connection. Prereq: notebook fabric/01 (slice + data-plane IPs).

## 1 · Configuration

In [ ]:
SLICE_NAME  = 'qfabric-bb84-2'      # same slice as notebooks fabric/01 / sequence/01 / 11F
STATION_IP  = '10.10.1.3'           # data-plane IP the bridge gives the switch node
NUM_PAIRS   = 4000                  # start small — WAN Cascade does many round trips;
                                    # raise to 20000 once a smoke run confirms the path
FIDELITY    = 0.95                  # per-link Werner parameter
DISTANCE_KM = 1.0                   # per-LINK distance -> pair loss
ATTENUATION = 0.2

## 2 · Load the slice

In [ ]:
import os, sys, json
from pathlib import Path

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
sys.path.insert(0, str(PROJECT_DIR)); sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
import deploy_fabric as df

fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show();
slice_obj.list_nodes();

## 3 · Ship code + station bridge + runtime (switch node included)

`setup_repeater_bridge` stops BMv2 (no photon plane in this protocol) and joins the
switch's two data-plane interfaces in a Linux bridge carrying the station IP, with
kernel forwarding so alice↔bob traffic still crosses the middle node. Re-run notebook fabric/02
/ `configure_switch` afterwards to restore BMv2 for the P4 experiments.

In [ ]:
df.upload_project(slice_obj)
df.setup_sequence_runtime(slice_obj, nodes=('alice', 'bob', 'switch'))
df.setup_repeater_bridge(slice_obj, station_ip=STATION_IP)

## 4 · BBM92 key over the swapped chain

In [ ]:
a_res, b_res, m_res = df.run_sequence_repeater(
    slice_obj, num_pairs=NUM_PAIRS, fidelity=FIDELITY,
    distance_km=DISTANCE_KM, attenuation=ATTENUATION,
    chain_mode='bbm92', station_ip=STATION_IP)

## 5 · CHSH across the WAN chain

In [ ]:
e_a, e_b, e_m = df.run_sequence_repeater(
    slice_obj, num_pairs=NUM_PAIRS, fidelity=FIDELITY,
    distance_km=DISTANCE_KM, attenuation=ATTENUATION,
    chain_mode='e91', station_ip=STATION_IP)

## 6 · Verify

In [ ]:
checks = []
def chk(n, ok, d=''):
    checks.append(ok); print(f"  [{'PASS' if ok else 'FAIL'}] {n}" + (f' — {d}' if d else ''))

chk('bbm92: both endpoints produced results', bool(a_res and b_res))
chk('bbm92: keys match bit-for-bit', a_res['key'] == b_res['key'] and a_res['key'] is not None)
chk('bbm92: QBER near the Werner law',
    abs(a_res['qber'] - a_res['qber_pred']) < 0.03,
    f"{a_res['qber']:.4f} vs law {a_res['qber_pred']:.4f}")
chk('station swapped every delivered attempt', m_res['swaps'] == a_res['delivered'])
chk('heralds crossed the station->bob link', e_b is None or b_res['heralds'] == m_res['heralds'])
chk('e91: CHSH violation over the WAN chain', e_a['chsh_s'] is not None and e_a['chsh_s'] > 2,
    f"S={e_a['chsh_s']:.3f} (law {e_a['chsh_pred']:.3f})")
print(f"\n{'ALL PASS' if all(checks) else 'SOME FAILED'} ({sum(checks)}/{len(checks)})")

## Notes

- Results land in `results/fabric_repeater_{alice,bob,repeater}.json`.
- **Herald-latency experiment**: `df.apply_classical_netem(slice_obj, delay_ms=50)`
  impairs the classical path; because the heralds ride their own station→bob
  connection, chain throughput now pays the herald RTT — measure time-to-key vs delay.
- Add `auth_key='...'` / `finite_key=True` to `run_sequence_repeater` for the
  authenticated / finite-key variants (finite-key needs large `NUM_PAIRS` on noisy
  chains — see notebook concepts/04).
- Restore the P4 path afterwards: re-run notebook fabric/02 (`configure_switch`) to bring BMv2
  back; the bridge is then unused.